In [11]:
import openslide
import os
import cv2
import numpy as np
import time
import libtiff
import pandas as pd


In [45]:
parent_path = "/rsrch5/home/trans_mol_path/cercan/data/acformer/qpproj_dummy"

images_folder = os.path.join(parent_path,"tiles")
csv_folder = os.path.join(parent_path,"predictions")
output_folder = os.path.join(parent_path,"predictions_stitched")
os.makedirs(output_folder, exist_ok =True)
img_list = os.listdir(images_folder)
wsi_search_folders=["/rsrch6/home/trans_mol_path/yuan_lab/TIER1/barretts/wsi/NU/NU_flow",
"/rsrch6/home/trans_mol_path/yuan_lab/TIER1/barretts/wsi/dysplasia/wholeDataset",
"/rsrch6/home/trans_mol_path/yuan_lab/TIER1/barretts/wsi/new_batch_slides/Barrett 05092024/renamedSlides"
]

In [53]:
def tiles2tiff(tile_folder, tile_size, image_size, output_folder, ext='.png', jpeg_compression=True, downsample = 2):




    img_name = os.path.basename(tile_folder)
    out_file = os.path.join(output_folder,img_name+'.tif')
    os.makedirs(output_folder, exist_ok=True)
    
    # n_tiles = len(glob.glob(os.path.join(tile_folder, '*.' + ext)))
    # tile_files = glob.glob(os.path.join(tile_path, '*.' + ext))
    tile_files = [f for f in os.listdir(img_folder) if f.endswith(ext)]

    if not tile_files:
        print('No tiles found in ' + tile_path + '!')
    else:
        tiff = libtiff.TIFF.open(out_file, mode='w')
        tiff.SetField(libtiff.libtiff_ctypes.TIFFTAG_IMAGEWIDTH, image_size[0]//downsample)
        tiff.SetField(libtiff.libtiff_ctypes.TIFFTAG_IMAGELENGTH, image_size[1]//downsample)
        tiff.SetField(libtiff.libtiff_ctypes.TIFFTAG_PHOTOMETRIC, libtiff.libtiff_ctypes.PHOTOMETRIC_RGB)
        tiff.SetField(libtiff.libtiff_ctypes.TIFFTAG_PLANARCONFIG, libtiff.libtiff_ctypes.PLANARCONFIG_CONTIG)
        tiff.SetField(libtiff.libtiff_ctypes.TIFFTAG_BITSPERSAMPLE, 8)
        tiff.SetField(libtiff.libtiff_ctypes.TIFFTAG_SAMPLESPERPIXEL, 3)
        tiff.SetField(libtiff.libtiff_ctypes.TIFFTAG_TILEWIDTH, tile_size[0]//downsample)
        tiff.SetField(libtiff.libtiff_ctypes.TIFFTAG_TILELENGTH, tile_size[1]//downsample)
        
        if jpeg_compression:
            tiff.SetField(libtiff.libtiff_ctypes.TIFFTAG_COMPRESSION, libtiff.libtiff_ctypes.COMPRESSION_JPEG)
            tiff.SetField(libtiff.libtiff_ctypes.TIFFTAG_JPEGQUALITY, 95)
        else:
            tiff.SetField(libtiff.libtiff_ctypes.TIFFTAG_COMPRESSION, libtiff.libtiff_ctypes.COMPRESSION_LZW)
        

        tile_files = removeEmptyCsv(tile_folder, csv_folder, tile_files)

        # Iterate through each tile file
        for i, tile_file in enumerate(tile_files):
            # print(f'Processing tile {tile_file}')
            print_progress_bar(i + 1, len(tile_files), prefix='Predicting tiles:', suffix='Complete', length=50)
            
            # Placeholder for extracting tile x, y coordinates from the filename
            # Replace the following lines with actual extraction logic
            x, y = (coord // downsample for coord in extract_tile_coordinates_from_filename(tile_file))

            im = getAnnotatedTile(tile_folder, csv_folder, tile_file, tile_size, downsample)
 
            # Write the tile to the TIFF image at the calculated position
            tiff.WriteTile(np.ascontiguousarray(im).ctypes.data, x , y , 0, 0)

        tiff.close()
        
def removeEmptyCsv(tile_folder, csv_folder, tile_files):
    final_tile_files = []
    for tile_file in tile_files:
        tile_export_path = os.path.join(csv_folder, os.path.basename(tile_folder),tile_file + '.csv')
        if os.path.getsize(tile_export_path) > 0:
            final_tile_files.append(tile_file)

    return final_tile_files

    
# Placeholder function to be implemented by you
def extract_tile_coordinates_from_filename(filename):
    x = int(filename.split('x=')[1].split(',')[0])
    y = int(filename.split('y=')[1].split(',')[0])

    return x, y

def getPredData(tile_folder, csv_folder, filename):

    tile_export_path = os.path.join(csv_folder, os.path.basename(tile_folder),filename + '.csv')
    data = pd.read_csv(tile_export_path, header=None, names=['x', 'y', 'class'])

    return data

def getAnnotatedTile(tile_folder, csv_folder, filename, tile_size, downsample):
    
    tile_export_path = os.path.join(csv_folder, os.path.basename(tile_folder),filename + '.csv')
    data = pd.read_csv(tile_export_path, header=None, names=['x', 'y', 'class'])
    tile_file_path=os.path.join(tile_folder,filename)
    tile_image= cv2.cvtColor(cv2.imread(tile_file_path), cv2.COLOR_BGR2RGB)

    for index, row in data.iterrows():
        # class_colors = {
    #     1: [255, 0, 0],  # Blue Lymph
    #     2: [0, 255, 0],  # Green Epithelium
    #     3: [0, 0, 255],  # Red Neut
    #     4: [100, 100, 100],  # Dark Gray Plasma
    #     5: [200, 200, 200],  # Light Gray Eo
    #     6: [50, 50, 50]  # Darker Gray Connective
    # }

        class_colors = {
            1: [255, 0, 0],  # Blue Lymph
            2: [0, 255, 0],  # Green Epithelium
            3: [255, 0, 255],  # Magenta Neut
            4: [255, 255, 0],  # Cyan Plasma
            5: [0, 0, 255],  # Red Eo
            6: [255, 255, 0]  # Yellow Connective
        }
        x, y, cell_class = row
        # x, y = x // downsample, y // downsample
        color = class_colors[cell_class]
        cv2.circle(tile_image, (x,y), radius=4, color=color, thickness=-1)
    
    height, width = (size // downsample for size in tile_size)
    
    tile_image= cv2.resize(tile_image, (height, width), interpolation=cv2.INTER_AREA)

    return tile_image

def get_wsi_dims(wsi_path):

    # Open the SVS file
    slide = openslide.OpenSlide(wsi_path)

    # Get the dimensions of the slide
    wsi_width, wsi_height = slide.dimensions

    # Close the slide
    slide.close()
    
    return wsi_width, wsi_height
    
def find_wsi_file(file_name, search_folders):
    """
    Search for the WSI file in the given folders and return the full path if found.
    :param file_name: The name of the WSI file to search for (with extension).
    :param search_folders: A list of folders to search in.
    :return: The full path of the WSI file if found, else None.
    """
    for folder in search_folders:
        ext= os.path.splitext(os.listdir(folder)[0])[1]
        file_path = os.path.join(folder, file_name+ext)
        if os.path.exists(file_path):
            break

    return file_path

def print_progress_bar(iteration, total, prefix='', suffix='', decimals=1, length=100, fill='█', print_end="\r"):
    percent = ("{0:." + str(decimals) + "f}").format(100 * (iteration / float(total)))
    filled_length = int(length * iteration // total)
    bar = fill * filled_length + '-' * (length - filled_length)
    print(f'\r{prefix} |{bar}| {percent}% {suffix}', end=print_end)
    if iteration == total:
        print()

In [54]:
for img in img_list:
    print('processing: '+ img)
    img_folder= os.path.join(images_folder,img)
    wsi_path = find_wsi_file(img, wsi_search_folders)
    wsi_width, wsi_height = get_wsi_dims(wsi_path)

    tiles2tiff(img_folder, tile_size=[256,256], image_size = [wsi_width, wsi_height], output_folder=output_folder, ext='png', jpeg_compression=True)

    # tiles_list = os.listdir(img_folder)

    # for tile_name in tiles_list:
    #     tile_path = os.path.join(img_folder,tile_name)
    #     tile_export_path = os.path.join(img_output, tile_name + '.csv')
    

processing: D1106-2020-11-16_15.31.14
Predicting tiles: |██████████████████████████████████████████████████| 100.0% Complete
processing: D1029-2020-11-17_15.30.08
Predicting tiles: |██████████████████████████████████████████████████| 100.0% Complete
processing: D4018
Predicting tiles: |██████████████████████████████████████████████████| 100.0% Complete
processing: D0044
Predicting tiles: |██████████████████████████████████████████████████| 100.0% Complete
processing: D0070
Predicting tiles: |██████████████████████████████████████████████████| 100.0% Complete
